# Prisoner’s Dilemma: from circuit to equilibrium

**A reproducible walkthrough for two, three, and four players.**

The question is whether any player can improve their expected payoff by changing
only their own strategy. This notebook reconstructs the pairwise-ZZ circuit in
the original research, checks its measurement probabilities, and solves its
pure-strategy Nash problem on the original angle grids.

**Result:** the implemented circuit has the same outcome probabilities as
independent classical randomization. The reference grids each have one
equilibrium behavior: all players defect. Every phase choice at that behavior
is tied. For k = 4, the full 18,974,736-profile angle domain is covered by
14,641 payoff representatives.

See [methods](../docs/methods.md) for the derivation and
[research notes](../docs/research-notes.md) for corrections to the original work.

## 1. Setup and conventions

Install from the repository root with `python -m pip install -e '.[notebook]'`.
No external dataset, cloud account, or quantum hardware is required. All results
below are freshly computed; no archived pickle is loaded.

Player i controls qubit i. In an action tuple, 0 means cooperate (C) and 1 means
defect (D). A fixed angle pair is one pure strategy, even if its measurement
outcome is probabilistic. These searches use a 1e-10 absolute payoff tolerance.

In [1]:
import platform
from itertools import product
from math import pi

import numpy as np
from IPython.display import Markdown, display

from quantum_pd import (
    __version__,
    classical_nash,
    classical_payoff,
    expected_payoff,
    probabilities,
    search_grid,
    statevector,
)

print(f"Python {platform.python_version()} | NumPy {np.__version__} | quantum-pd {__version__}")

Python 3.12.14 | NumPy 2.5.3 | quantum-pd 0.1.0


## 2. Recover the classical baseline

Two players receive the standard Prisoner’s Dilemma payoffs. The unique pure
Nash equilibrium is (D, D): each player would lose payoff by switching to C.

In [2]:
rows = ["| Player 0 | Player 1 | Payoffs |", "|---|---|---|"]
for actions in product([0, 1], repeat=2):
    labels = ["D" if bit else "C" for bit in actions]
    payoff = tuple(int(x) for x in classical_payoff(actions))
    rows.append(f"| {labels[0]} | {labels[1]} | {payoff} |")
display(Markdown("\n".join(rows)))
assert classical_nash(2) == [{"actions": [1, 1], "payoffs": [1.0, 1.0]}]

| Player 0 | Player 1 | Payoffs |
|---|---|---|
| C | C | (3, 3) |
| C | D | (0, 5) |
| D | C | (5, 0) |
| D | D | (1, 1) |

### Keep multiplayer payoff rules separate

The circuit notebooks use a collective payoff: all-C pays 3, all-D pays 1;
otherwise a defector gets 5 and a cooperator gets 0. The classical notebooks
sum pairwise PD payoffs instead. These define different games for k > 2.
The reconstruction supports both, with matched classical baselines.

In [3]:
for model in ("collective", "pairwise"):
    payoff = classical_payoff([1, 0, 0], model)
    print(f"{model:10s} (D, C, C): {payoff.tolist()}")

collective (D, C, C): [5.0, 0.0, 0.0]
pairwise   (D, C, C): [10.0, 3.0, 3.0]


## 3. Check the actual circuit

The original CX–RZ–CX sequence implements a diagonal ZZ rotation. Its action on
the initial all-zero state is a global phase, and its inverse immediately before
measurement changes no basis probabilities. With local `Ry(theta)` followed by
`Rz(phi)`, the probability that player i defects is `sin²(theta_i / 2)`.

The following asymmetric case catches the original player-ordering error:
player 0 defects and player 1 cooperates, so the payoff must be (5, 0).

In [4]:
angles = [(pi, 0), (0, 0)]
assert np.allclose(probabilities(angles), [0, 1, 0, 0])
assert np.allclose(expected_payoff(angles), [5, 0])
print("Displayed basis state: 01 | player actions: (D, C) | payoffs: (5, 0)")

Displayed basis state: 01 | player actions: (D, C) | payoffs: (5, 0)


### Check phase invariance at an interior strategy profile

This numerical check compares the reduced probability formula with a complex
statevector that retains the ZZ and local phase rotations. The test suite also
checks both against Qiskit’s gate-level simulation; this notebook needs only NumPy.

In [5]:
angles = np.array([[0.4, 0.2], [1.1, 0.7], [2.6, 1.3], [2.0, 0.1]])
reference = probabilities(angles)
for gamma in (0, 0.31, pi / 2):
    error = np.max(np.abs(np.abs(statevector(angles, gamma)) ** 2 - reference))
    assert error < 1e-12
    print(f"gamma = {gamma:.3f}: max probability error = {error:.2e}")
assert np.isclose(reference.sum(), 1.0)

gamma = 0.000: max probability error = 1.11e-16
gamma = 0.310: max probability error = 1.11e-16
gamma = 1.571: max probability error = 8.33e-17


## 4. Solve the original angle grids

At a fixed theta profile, all phase choices have identical payoffs and incentives.
The search evaluates one representative per theta profile and counts every phase
choice. It retains all tied best responses and returns each player’s maximum
profitable deviation (regret).

The table below reports freshly recomputed results for both payoff rules. The
phase multiplicity counts equivalent angle profiles, not different behaviors.

In [6]:
results = []
for model in ("collective", "pairwise"):
    for players in (2, 3, 4):
        theta, phi = (11, 6) if players == 4 else (15, 8)
        result = search_grid(players, theta_points=theta, phi_points=phi, model=model)
        results.append(result)

rows = [
    "| Rule | k | Full profiles | Representatives | Tied equilibrium profiles | Payoff/player |",
    "|---|---:|---:|---:|---:|---:|",
]
for result in results:
    (equilibrium,) = result["equilibrium_classes"]
    assert equilibrium["theta_indices"] == [result["theta_points"] - 1] * result["players"]
    assert max(equilibrium["unilateral_regrets"]) <= result["absolute_tolerance"]
    rows.append(
        f"| {result['payoff_model']} | {result['players']} | "
        f"{result['full_angle_profiles']:,} | {result['evaluated_representatives']:,} | "
        f"{result['equilibrium_angle_profiles']:,} | {equilibrium['payoffs'][0]:.0f} |"
    )
display(Markdown("\n".join(rows)))

| Rule | k | Full profiles | Representatives | Tied equilibrium profiles | Payoff/player |
|---|---:|---:|---:|---:|---:|
| collective | 2 | 14,400 | 225 | 64 | 1 |
| collective | 3 | 1,728,000 | 3,375 | 512 | 1 |
| collective | 4 | 18,974,736 | 14,641 | 1,296 | 1 |
| pairwise | 2 | 14,400 | 225 | 64 | 1 |
| pairwise | 3 | 1,728,000 | 3,375 | 512 | 2 |
| pairwise | 4 | 18,974,736 | 14,641 | 1,296 | 3 |

## 5. Inspect an equilibrium certificate

For the collective k = 4 game, the representative is theta = pi and phi = 0
for every player. Each payoff is 1; no unilateral grid deviation improves it.
There are 6⁴ = 1,296 equally good phase assignments.

In [7]:
four_player = next(r for r in results if r["players"] == 4 and r["payoff_model"] == "collective")
(equilibrium,) = four_player["equilibrium_classes"]
print("Theta / pi:", [round(a[0] / pi, 6) for a in equilibrium["representative_angles"]])
print("Payoffs:", equilibrium["payoffs"])
print("Unilateral regrets:", equilibrium["unilateral_regrets"])
print("Represented phase profiles:", equilibrium["represented_angle_profiles"])

Theta / pi: [1.0, 1.0, 1.0, 1.0]
Payoffs: [1.0, 1.0, 1.0, 1.0]
Unilateral regrets: [0.0, 0.0, 0.0, 0.0]
Represented phase profiles: 1296


## Interpretation

The circuit explored here does not change the equilibrium behavior from the
matched classical game. Its diagonal phases do not affect measured payoffs.
The phase reduction is therefore an exact property of this model, not a claimed
quantum computing speedup.

The finite-grid computation and the analytical argument are separate evidence.
The grid checks all listed alternatives within tolerance. For the continuous
angle domain, defection is strictly dominant under either payoff rule, so
expected payoff is maximized at theta = pi regardless of opponents; see the
[dominance derivation](../docs/methods.md#search-and-equilibrium-certificate).

A different entangler or strategy family requires a new derivation and new
circuit checks. The original notebooks and saved outputs remain in
[the archive](../archive/README.md); the full historical k = 4 pickle was not
available. This walkthrough is an AI-assisted reconstruction with fresh results,
not a recreation of that missing run.